# ComfyUI on Google Colab GPU — official Agent Panel

Runs the `tienoho/ComfyUI` fork on a Colab GPU with the **first-party Agent Panel only**. No separate agent, chat UI, external public tunnel, or vendor API patching.

**Before running:** Runtime → Change runtime type → GPU. Colab resources and session duration are not guaranteed. Colab's browser proxy may not support every Agent/WebSocket/auth flow; verify with the native probe and real browser UAT. Never share your Colab session URL or upload credentials into the notebook.

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'GPU not available: Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))
ROOT = Path('/content/ComfyUI')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/tienoho/ComfyUI.git',str(ROOT)],check=True)
else:
    print('Existing checkout preserved; review and update manually if needed')
print('Repo:',ROOT)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements.txt')],check=True)
# Official first-party agent is optional; pin the reviewed pre-release for reproducibility.
ENABLE_AGENT = True
if ENABLE_AGENT:
    subprocess.run([sys.executable,'-m','pip','install','--pre','comfy-agent[go-nonfree]==0.1.0rc8'],check=True)
    subprocess.run([sys.executable,'-m','pip','check'],check=True)
print('Dependencies installed')

## Optional: Google Drive model storage

Mount Drive only if needed. Place your legally obtained checkpoint files in `MyDrive/ComfyUI/models/checkpoints` (and similarly for `loras`, `vae`, `diffusion_models`). Files remain private in your Drive. Large models may load slower from Drive; copy to `/content/ComfyUI/models/` for better inference speed.

In [ ]:
USE_DRIVE_MODELS = False
if USE_DRIVE_MODELS:
    from google.colab import drive
    drive.mount('/content/drive')
    source = Path('/content/drive/MyDrive/ComfyUI/models')
    if not source.is_dir():
        raise FileNotFoundError(f'Create model folder first: {source}')
    import shutil
    for category in ('checkpoints','loras','vae','diffusion_models','text_encoders','clip'):
        origin = source/category
        if origin.is_dir():
            dest = ROOT/'models'/category
            dest.mkdir(parents=True,exist_ok=True)
            for item in origin.iterdir():
                if item.is_file():
                    shutil.copy2(item,dest/item.name)
    print('Models copied into Colab local disk')

In [ ]:
import subprocess, sys, time, urllib.request, json
LOG = open('/content/comfyui-colab.log','a',buffering=1)
command = [sys.executable,'main.py','--listen','127.0.0.1','--port','8188']
if ENABLE_AGENT:
    command.append('--enable-agent')
process = subprocess.Popen(command,cwd=str(ROOT),stdout=LOG,stderr=subprocess.STDOUT,start_new_session=True)
print('ComfyUI PID:',process.pid)
for attempt in range(90):
    if process.poll() is not None:
        raise RuntimeError('ComfyUI exited early; inspect /content/comfyui-colab.log')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/features',timeout=3) as response:
            features = json.load(response)
        print('Feature response:',features)
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError('ComfyUI did not start; inspect /content/comfyui-colab.log')

In [ ]:
from google.colab import output
# Uses Colab's session-scoped proxy, NOT a public ngrok/cloudflared tunnel.
# Browser WebSocket / Agent authentication compatibility requires real UAT.
output.serve_kernel_port_as_window(8188)

In [ ]:
import subprocess, sys
if ENABLE_AGENT:
    subprocess.run([sys.executable,'script_examples/check_native_agent.py','--wait-seconds','90'],cwd=str(ROOT),check=False)
print('Recent log lines:')
print(''.join(Path('/content/comfyui-colab.log').read_text(errors='replace').splitlines(keepends=True)[-45:]))

## Shutdown and security

The UI runs behind Colab's session proxy, but **Colab is not a production hosting environment**. Keep the notebook private, do not add `--listen 0.0.0.0`, avoid public tunnels, and never place Comfy credentials or API keys in notebook output. If the first-party Agent cannot authenticate through the Colab proxy, use regular ComfyUI GPU workflows and inspect logs; do not bypass sign-in or consent.

In [ ]:
STOP_SERVER = False
if STOP_SERVER and process.poll() is None:
    process.terminate()
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.kill()
    print('ComfyUI stopped')